# Stage 2 — Faithfulness Results

This notebook analyses the faithfulness results from Stage 2 expert steering experiments.

**Three datasets**:
- **Counterfactual** (`FaithEval-Counterfactual`): MCQ where the context contradicts training knowledge. A faithful model follows the context. Metric: accuracy on the context-supported answer.
- **Unanswerable** (`FaithEval-Unanswerable`): Questions the context cannot answer. A faithful model abstains. Metric: proportion of correct abstentions.
- **SQuAD control** (`MCTest`): Standard QA where context and knowledge agree. Should be unaffected by steering. Metric: accuracy.

**Steering direction**: faithfulness negative — suppress confabulation-preferred experts.

**Conditions**: `baseline` (no steering) vs `hard` (full expert deactivation).

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display as ipy_display

sns.set_theme(style='whitegrid', font='serif')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

RESULTS_DIR = '/users/sc23jc3/projects/Individual-Project-25-26/stage2/results'
OUT_DIR = 'figures'
os.makedirs(OUT_DIR, exist_ok=True)

FAITH_TASKS = ['faith_cf', 'faith_un', 'faith_mc']
CONDITIONS  = ['baseline', 'hard', 'soft']
TASK_LABELS = {
    'faith_cf': 'Counterfactual\n(FaithEval)',
    'faith_un': 'Unanswerable\n(FaithEval)',
    'faith_mc': 'SQuAD Control\n(MCTest)',
}


def load_cond(task, cond):
    path = os.path.join(RESULTS_DIR, task, f"{cond}.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        d = json.load(f)
    d['_by_idx'] = {r['idx']: r for r in d.get('records', [])}
    return d


data = {}
for task in FAITH_TASKS:
    data[task] = {}
    for cond in CONDITIONS:
        d = load_cond(task, cond)
        data[task][cond] = d
        if d:
            print(f"  {task}/{cond}: n={d['n_complete']}, accuracy={d.get('accuracy', float('nan')):.3f}")
        else:
            print(f"  {task}/{cond}: NOT FOUND")

## 1. Accuracy: Baseline vs Hard Steering

In [ ]:
def get_acc(task, cond):
    d = data[task][cond]
    return d['accuracy'] if d else float('nan')

task_keys   = FAITH_TASKS
task_labels = [TASK_LABELS[t] for t in task_keys]
baselines   = [get_acc(t, 'baseline') for t in task_keys]
hards       = [get_acc(t, 'hard')     for t in task_keys]
softs       = [get_acc(t, 'soft')     for t in task_keys]

x = np.arange(len(task_labels))
w = 0.22

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w, baselines, width=w, label='Baseline',      color='#7f8c8d', alpha=0.85)
ax.bar(x,     hards,     width=w, label='Hard steering', color='#27ae60', alpha=0.85)
ax.bar(x + w, softs,     width=w, label='Soft steering', color='#f39c12', alpha=0.85)

for i, (b, h, s) in enumerate(zip(baselines, hards, softs)):
    for val, xpos, color in [(h, x[i], '#27ae60'), (s, x[i] + w, '#f39c12')]:
        if not np.isnan(b) and not np.isnan(val):
            delta = val - b
            sign = '+' if delta >= 0 else ''
            ax.annotate(f'{sign}{delta:.3f}', xy=(xpos, val), xytext=(0, 4),
                        textcoords='offset points', ha='center', fontsize=9,
                        color=color, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(task_labels)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.15)
ax.set_title('Faithfulness Results: Baseline vs Hard vs Soft Expert Steering')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'faithfulness_results.png'), dpi=300, bbox_inches='tight')
plt.show()

## 2. Numerical Summary

In [ ]:
print(f"{'Dataset':35s} {'Baseline':>10} {'Hard':>10} {'Δ Hard':>10} {'Soft':>10} {'Δ Soft':>10}")
print('-' * 85)
for task in task_keys:
    b  = get_acc(task, 'baseline')
    h  = get_acc(task, 'hard')
    s  = get_acc(task, 'soft')
    dh = h - b if not (np.isnan(b) or np.isnan(h)) else float('nan')
    ds = s - b if not (np.isnan(b) or np.isnan(s)) else float('nan')
    label = TASK_LABELS[task].replace('\n', ' ')
    print(f"{label:35s} {b:10.3f} {h:10.3f} {dh:+10.3f} {s:10.3f} {ds:+10.3f}")

## 3. Mismatch Inspection

Records where steering flips the correctness label — baseline wrong but steered correct (or vice versa).
Full prompt, context, and both responses side by side.

In [ ]:
_CORRECT_BADGE   = '<span style="background:#27ae60;color:#fff;padding:2px 8px;border-radius:3px;font-weight:700;font-size:12px">CORRECT</span>'
_INCORRECT_BADGE = '<span style="background:#c0392b;color:#fff;padding:2px 8px;border-radius:3px;font-weight:700;font-size:12px">WRONG</span>'

_STYLE = """<style>
.mm { font-family: Arial,sans-serif; color:#111 }
.mm h3 { margin-top:24px; border-bottom:2px solid #2c3e50; padding-bottom:4px; font-weight:700 }
.mm .card { border:1px solid #bdc3c7; border-radius:6px; margin:12px 0; padding:14px; background:#fff }
.mm .lbl  { font-size:12px; font-weight:700 }
.mm .prompt { background:#eaf0fb; border-left:3px solid #27ae60; padding:8px 10px; margin-top:4px;
              white-space:pre-wrap; font-size:12px; font-family:monospace; line-height:1.45; color:#111 }
.mm .grid { display:grid; grid-template-columns:1fr 1fr; gap:12px }
.mm .box  { background:#f8f9fa; border:1px solid #dee2e6; border-radius:4px; padding:10px;
            max-height:220px; overflow-y:auto; white-space:pre-wrap;
            font-family:monospace; font-size:12px; line-height:1.45; color:#111 }
</style>"""

def _esc(t):
    return str(t).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")

def _cbadge(correct):
    return _CORRECT_BADGE if correct else _INCORRECT_BADGE

def render_faith_mismatches(task_label, cond, mismatches):
    if not mismatches:
        ipy_display(HTML(f'<p><em>No mismatches for {task_label} / {cond}.</em></p>'))
        return
    parts = [_STYLE, f'<div class="mm"><h3>{task_label} &#9656; baseline vs <b>{cond}</b> &nbsp;({len(mismatches)} mismatches)</h3>']
    for m in mismatches:
        arrow = f'{_cbadge(m["base_correct"])} &rarr; {_cbadge(m["steered_correct"])}'
        gold  = _esc(m.get("gold") or "")
        parts.append(f"""<div class="card">
  <div style="margin-bottom:8px;font-weight:600">idx {m['idx']} &nbsp; {arrow} &nbsp; gold: <code>{gold}</code></div>
  <div style="margin-bottom:10px"><b class="lbl">PROMPT</b><div class="prompt">{_esc(m['prompt'])}</div></div>
  <div class="grid">
    <div><b class="lbl">BASELINE</b><div class="box">{_esc(m['base_response'])}</div></div>
    <div><b class="lbl">{cond.upper()}</b><div class="box" style="background:#fef9e7">{_esc(m['steer_response'])}</div></div>
  </div></div>""")
    parts.append('</div>')
    ipy_display(HTML('\n'.join(parts)))


for task in FAITH_TASKS:
    base = data[task]['baseline']
    if not base:
        print(f"{task}: baseline not loaded, skipping")
        continue
    for cond in ['hard', 'soft']:
        steered = data[task][cond]
        if not steered:
            print(f"{task}/{cond}: not loaded")
            continue
        if 'mismatch_indices' not in steered:
            print(f"{task}/{cond}: mismatch annotation not available (run baseline first)")
            continue
        mismatches = [
            {
                'idx':             idx,
                'base_correct':    base['_by_idx'][idx]['correct'],
                'steered_correct': steered['_by_idx'][idx]['correct'],
                'gold':            base['_by_idx'][idx].get('gold'),
                'prompt':          base['_by_idx'][idx]['prompt'],
                'base_response':   base['_by_idx'][idx]['response'],
                'steer_response':  steered['_by_idx'][idx]['response'],
            }
            for idx in steered['mismatch_indices']
            if idx in base['_by_idx'] and idx in steered['_by_idx']
        ]
        render_faith_mismatches(TASK_LABELS[task].replace('\n', ' '), cond, mismatches)

## Discussion

*(Fill in after examining the results.)*

**Counterfactual**:
- If hard steering raises accuracy, the model is following context more faithfully after suppressing confabulation experts. Strong causal finding.

**Unanswerable**:
- If hard steering raises abstention accuracy, the model is becoming more appropriately agnostic — not hallucinating answers when the context is insufficient.

**SQuAD control**:
- This should ideally remain flat. A significant drop would indicate steering degrades general reading comprehension — a sign the intervention is too crude rather than specific.
- A small drop is acceptable; a large drop undermines the causal interpretation.

**Key question**: does faithfulness improve on the test datasets *without* degrading the control?